# BLUE fusion of multiple camera views + cross-correlation synchronization

Standalone notebook: takes the per-view/per-target results already computed by the
digitization notebook (`mc_result_{View}_{Target}.npz`) and produces:
1. The time lag between View_1 and View_2 (velocity cross-correlation, not displacement).
2. The optimal (BLUE) per-frame fusion of both views, using each view's Monte Carlo covariance.
3. Figures and result files, saved in an organized way under `output/Fusion_BLUE/`.

**Requirements:** `numpy`, `pandas`, `matplotlib`, `scipy`. Run the digitization notebook
first for every (view, target) pair — this notebook only reads its `mc_result_*.npz` outputs.

## 1. Configuration (required)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags
from matplotlib.patches import Ellipse

# ============================  PARAMETERS  ============================
# EDIT THIS: base working folder (same one used by the digitization notebook).
BASE_PATH = "/path/to/your/project"

VIEWS   = ["Vista_1", "Vista_2"]  # view names, in the order they get synced (2nd relative to 1st)
TARGETS = ["R"]                   # target(s) to fuse -- this project only tracks the right TDP (R)

PAD_SEC   = 5.0    # physical padding for the fused velocity (must match the rest of the pipeline)
MAX_LAG_S = 2.0    # search range for the lag between views (s)
DPI_SAVE  = 300

# --- Paths (follow the tidy output/{View}_{Target}/... layout) ---
def mc_result_path(view, target):
    return os.path.join(BASE_PATH, "output", f"{view}_{target}", f"mc_result_{view}_{target}.npz")

def raw_trace_path(view, target):
    return os.path.join(BASE_PATH, "output", f"{view}_{target}", f"{view}_{target}_raw_trace.txt")

FUSION_DIR = os.path.join(BASE_PATH, "output", "Fusion_BLUE")
FUSION_FIGURES_DIR = os.path.join(FUSION_DIR, "figures")
os.makedirs(FUSION_FIGURES_DIR, exist_ok=True)
_FUSION_MANIFEST_PATH = os.path.join(FUSION_DIR, "figures_manifest.json")

print("Example paths (Vista_2, R):")
print(" ", mc_result_path("Vista_2", "R"))
print(" ", raw_trace_path("Vista_2", "R"))
print("Fusion output folder:", FUSION_DIR)


## 2. Utilities (run once)

In [ ]:
def save_fig_fusion(fig, fig_id, caption, dpi=DPI_SAVE):
    """Save a figure to FUSION_FIGURES_DIR/{fig_id}.png and register
    (fig_id, path, caption) in output/Fusion_BLUE/figures_manifest.json."""
    path = os.path.join(FUSION_FIGURES_DIR, f'{fig_id}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    entry = {'fig_id': fig_id, 'path': path, 'caption': caption}
    manifest = []
    if os.path.exists(_FUSION_MANIFEST_PATH):
        with open(_FUSION_MANIFEST_PATH, encoding='utf-8') as f:
            manifest = json.load(f)
    manifest = [m for m in manifest if m['fig_id'] != fig_id] + [entry]
    with open(_FUSION_MANIFEST_PATH, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=1, ensure_ascii=False)
    print(f'Figure saved: {path}')


def load_mc_result(view, target, lag=0.0):
    """Load mc_result_{view}_{target}.npz. t is shifted to start at 0 (relative to its own
    digitization window) and then offset by `lag` (used to align View_2 relative to View_1
    once the lag has been estimated in section 4)."""
    path = mc_result_path(view, target)
    d = np.load(path)
    t = d['t_serie'] - d['t_serie'][0] + lag
    return dict(t=t, xy=d['rel_serie'], xy_raw=d['rel_serie_raw'], C_frames=d['C_frames'])


def load_raw_trace(view, target):
    """Load the raw trace (t_s, east_m, north_m) saved before the Monte Carlo/filtering step,
    for diagnostic overlays only -- does not take part in the BLUE fusion itself."""
    path = raw_trace_path(view, target)
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, sep=r'\s+')
    t = df.iloc[:, 0].values
    xy = df.iloc[:, 1:3].values
    return dict(t=t - t[0], xy=xy)


def interp_cov(C_frames, t_src, t_dst):
    """Linearly interpolate each covariance component (2x2 -> 4 scalar series) and re-impose
    exact symmetry (per-component interpolation can break it through rounding)."""
    n = len(t_dst)
    C_interp = np.empty((n, 2, 2))
    for a in range(2):
        for b in range(2):
            C_interp[:, a, b] = np.interp(t_dst, t_src, C_frames[:, a, b])
    return 0.5 * (C_interp + np.transpose(C_interp, (0, 2, 1)))


def fuse_blue_per_frame(x1, C1, x2, C2):
    """Per-frame BLUE fusion (full 2x2 covariance matrix, not just diagonal variances).
    x1, x2: (n,2). C1, C2: (n,2,2)."""
    n = len(x1)
    x_fused = np.empty_like(x1)
    C_fused = np.empty_like(C1)
    for i in range(n):
        W1 = np.linalg.inv(C1[i])
        W2 = np.linalg.inv(C2[i])
        Ci = np.linalg.inv(W1 + W2)
        C_fused[i] = Ci
        x_fused[i] = Ci @ (W1 @ x1[i] + W2 @ x2[i])
    return x_fused, C_fused


def pad_only(x, t, pad_sec):
    """Physical padding (repeats the first/last real value) with NO filtering -- only to avoid
    edge artifacts from np.gradient when computing velocity."""
    dt_med = np.median(np.diff(t))
    n_pad = int(round(pad_sec / dt_med))
    start_block = np.repeat(x[0:1], n_pad, axis=0)
    end_block   = np.repeat(x[-1:], n_pad, axis=0)
    x_padded = np.concatenate([start_block, x, end_block], axis=0)
    t_padded = np.concatenate([
        t[0] - dt_med * np.arange(n_pad, 0, -1), t,
        t[-1] + dt_med * np.arange(1, n_pad + 1)])
    return x_padded, t_padded, (t_padded[0], t[0]), (t[-1], t_padded[-1]), n_pad


def add_cov_ellipse(ax, mean, cov, nsig=2, **kw):
    vals, vecs = np.linalg.eigh(cov)
    ang = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    ax.add_patch(Ellipse(mean, 2 * nsig * np.sqrt(vals[1]), 2 * nsig * np.sqrt(vals[0]), angle=ang, **kw))

print("Utilities loaded.")


## 3. Load results per view/target

In [ ]:
assert len(VIEWS) == 2, "This notebook is written for exactly 2 views (VIEWS[0] is the reference)."
VIEW_REF, VIEW_2 = VIEWS

data_raw = {}   # data_raw[(view, target)] -> dict(t, xy, xy_raw, C_frames), lag NOT applied yet
for view in VIEWS:
    for target in TARGETS:
        data_raw[(view, target)] = load_mc_result(view, target)
        d = data_raw[(view, target)]
        print(f'{view} {target}: {len(d["t"])} samples, t in [{d["t"][0]:.2f}, {d["t"][-1]:.2f}] s')


## 4. Time synchronization between views (velocity cross-correlation)

Displacement magnitude gives a broad-lobed correlation (not very informative for the lag);
velocity (the derivative) has a more localized peak in time and gives a sharper correlation.
Since both targets come from the same pair of videos, the true lag is a single shared value:
the (normalized) correlation curves of all targets are summed before finding the joint
maximum, instead of averaging per-target lag estimates.

In [ ]:
def corr_curve(t1, s1_raw, t2, s2_raw, max_lag_s, dt):
    t_min = max(t1[0], t2[0]) - max_lag_s
    t_max = min(t1[-1], t2[-1]) + max_lag_s
    t_grid = np.arange(t_min, t_max, dt)

    s1 = np.interp(t_grid, t1, s1_raw, left=np.nan, right=np.nan)
    s2 = np.interp(t_grid, t2, s2_raw, left=np.nan, right=np.nan)
    valid = ~(np.isnan(s1) | np.isnan(s2))
    if valid.sum() < 10:
        raise ValueError('Too little overlap to estimate the lag; check MAX_LAG_S.')

    s1c, s2c = s1.copy(), s2.copy()
    s1c[~valid] = 0; s2c[~valid] = 0
    s1c -= s1c[valid].mean(); s2c -= s2c[valid].mean()
    s1c /= (s1c[valid].std() + 1e-12); s2c /= (s2c[valid].std() + 1e-12)

    corr = correlate(s1c, s2c, mode='full')
    lags = correlation_lags(len(s1c), len(s2c), mode='full') * dt
    mask = np.abs(lags) <= max_lag_s
    return lags[mask], corr[mask]


def peak_width(lags, corr, frac=0.9):
    thresh = corr.max() * frac
    idxs = np.where(corr >= thresh)[0]
    return lags[idxs[-1]] - lags[idxs[0]]


def subsample_peak(lags, corr):
    """3-point quadratic fit around the maximum -> sub-frame-precision lag."""
    i = np.argmax(corr)
    if i == 0 or i == len(corr) - 1:
        return lags[i]
    y0, y1, y2 = corr[i - 1], corr[i], corr[i + 1]
    denom = y0 - 2 * y1 + y2
    if abs(denom) < 1e-12:
        return lags[i]
    delta = 0.5 * (y0 - y2) / denom
    dt_local = lags[i + 1] - lags[i]
    return lags[i] + delta * dt_local


dt_common_lag = min(np.median(np.diff(data_raw[(v, c)]['t'])) for v in VIEWS for c in TARGETS)

vel_by_target = {}
for target in TARGETS:
    d1, d2 = data_raw[(VIEW_REF, target)], data_raw[(VIEW_2, target)]
    v1 = np.gradient(np.linalg.norm(d1['xy'], axis=1), d1['t'])
    v2 = np.gradient(np.linalg.norm(d2['xy'], axis=1), d2['t'])
    vel_by_target[target] = (d1['t'], v1, d2['t'], v2)

corr_sum = None
for target, (t1, v1, t2, v2) in vel_by_target.items():
    lags, corr = corr_curve(t1, v1, t2, v2, MAX_LAG_S, dt_common_lag)
    corr_sum = corr.copy() if corr_sum is None else corr_sum + corr

LAG_CONSENSUS = subsample_peak(lags, corr_sum)
w90_joint = peak_width(lags, corr_sum, 0.90)
print(f'Single lag ({VIEW_2} relative to {VIEW_REF}), combined velocity {"+".join(TARGETS)}: '
      f'{LAG_CONSENSUS:+.4f} s')
print(f'Joint peak width at 90%: {w90_joint:.3f} s '
      f'-> actual lag resolution (do not trust more precision than this)')

fig_lag, axs_lag = plt.subplots(1, len(TARGETS), figsize=(5.5 * len(TARGETS), 4), squeeze=False)
axs_lag = axs_lag[0]
for ax, target in zip(axs_lag, TARGETS):
    t1, v1, t2, v2 = vel_by_target[target]
    lags_i, corr_i = corr_curve(t1, v1, t2, v2, MAX_LAG_S, dt_common_lag)
    lag_i = subsample_peak(lags_i, corr_i)
    w90_i = peak_width(lags_i, corr_i, 0.90)
    print(f'  Target {target}: individual lag = {lag_i:+.4f} s (90% width = {w90_i:.3f} s)')

    ax.plot(lags_i, corr_i, label=f'individual lag = {lag_i:+.3f} s')
    ax.axvline(LAG_CONSENSUS, color='r', ls='--', label=f'joint lag = {LAG_CONSENSUS:+.3f} s')
    ax.set_title(f'Velocity cross-correlation -- Target {target}')
    ax.set_xlabel('lag (s)'); ax.set_ylabel('correlation'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig_fusion(fig_lag, 'fig08_synchronization',
                f'Velocity cross-correlation between {VIEW_REF} and {VIEW_2}, per target, '
                'with the joint lag and the individual ones.')

print('\nIf the individual lag of each target differs a lot from the joint one, or the '
      'correlation peak is wide/unclear, check MAX_LAG_S and tracking quality before '
      'trusting LAG_CONSENSUS.')


## 5. Optimal (BLUE) per-frame fusion

In [ ]:
# Apply the estimated lag: VIEW_REF stays as-is, VIEW_2 is shifted by LAG_CONSENSUS.
data_synced = {}
for target in TARGETS:
    data_synced[(VIEW_REF, target)] = load_mc_result(VIEW_REF, target, lag=0.0)
    data_synced[(VIEW_2, target)]   = load_mc_result(VIEW_2, target, lag=LAG_CONSENSUS)

fused_results = {}
for target in TARGETS:
    d1, d2 = data_synced[(VIEW_REF, target)], data_synced[(VIEW_2, target)]
    print(f'Target {target}: {VIEW_REF} t in [{d1["t"][0]:.2f}, {d1["t"][-1]:.2f}] | '
          f'{VIEW_2} t in [{d2["t"][0]:.2f}, {d2["t"][-1]:.2f}]')

    t_min = max(d1['t'][0], d2['t'][0])
    t_max = min(d1['t'][-1], d2['t'][-1])
    dt_common = min(np.median(np.diff(d1['t'])), np.median(np.diff(d2['t'])))
    t_common = np.arange(t_min, t_max + dt_common / 2, dt_common)

    x1 = np.column_stack([np.interp(t_common, d1['t'], d1['xy'][:, k]) for k in range(2)])
    x2 = np.column_stack([np.interp(t_common, d2['t'], d2['xy'][:, k]) for k in range(2)])
    C1 = interp_cov(d1['C_frames'], d1['t'], t_common)
    C2 = interp_cov(d2['C_frames'], d2['t'], t_common)

    x_fused, C_fused = fuse_blue_per_frame(x1, C1, x2, C2)
    fused_results[target] = dict(t=t_common, x1=x1, x2=x2, x_fused=x_fused, C1=C1, C2=C2, C_fused=C_fused)

    s1E, s1N = np.sqrt(C1[:, 0, 0]).mean(), np.sqrt(C1[:, 1, 1]).mean()
    s2E, s2N = np.sqrt(C2[:, 0, 0]).mean(), np.sqrt(C2[:, 1, 1]).mean()
    sfE, sfN = np.sqrt(C_fused[:, 0, 0]).mean(), np.sqrt(C_fused[:, 1, 1]).mean()
    print(f'  mean sigma (cm) -- {VIEW_REF}: E={s1E*100:.2f} N={s1N*100:.2f} | '
          f'{VIEW_2}: E={s2E*100:.2f} N={s2N*100:.2f} | Fused: E={sfE*100:.2f} N={sfN*100:.2f}')


## 6. Shared axis limits across targets (for a fair comparison)

In [ ]:
pos_lims = {}
for col in range(2):
    vals = np.concatenate([np.concatenate([fused_results[c]['x1'][:, col],
                                            fused_results[c]['x2'][:, col],
                                            fused_results[c]['x_fused'][:, col]])
                            for c in TARGETS])
    pad = 0.05 * (vals.max() - vals.min())
    pos_lims[col] = (vals.min() - pad, vals.max() + pad)

sigma_lims = {}
for col in range(2):
    vals = np.concatenate([np.concatenate([np.sqrt(fused_results[c]['C1'][:, col, col]),
                                            np.sqrt(fused_results[c]['C2'][:, col, col]),
                                            np.sqrt(fused_results[c]['C_fused'][:, col, col])]) * 100
                            for c in TARGETS])
    sigma_lims[col] = (0, vals.max() * 1.05)

xy_all = np.concatenate([np.concatenate([fused_results[c]['x1'], fused_results[c]['x2'],
                                          fused_results[c]['x_fused']]) for c in TARGETS])
x_pad = 0.05 * (xy_all[:, 0].max() - xy_all[:, 0].min())
y_pad = 0.05 * (xy_all[:, 1].max() - xy_all[:, 1].min())
plan_xlim = (xy_all[:, 0].min() - x_pad, xy_all[:, 0].max() + x_pad)
plan_ylim = (xy_all[:, 1].min() - y_pad, xy_all[:, 1].max() + y_pad)

vel_all = []
for target in TARGETS:
    fr = fused_results[target]
    x_fused_pad, t_fused_pad, _, _, _ = pad_only(fr['x_fused'], fr['t'], PAD_SEC)
    vel_all.append(np.gradient(x_fused_pad, np.median(np.diff(t_fused_pad)), axis=0))
vel_all = np.concatenate(vel_all)
v_pad = 0.05 * (vel_all.max() - vel_all.min())
vel_ylim = (vel_all.min() - v_pad, vel_all.max() + v_pad)

print('Limits computed and shared across', TARGETS)


## 7. Figures

In [ ]:
# --- Components: View_1, View_2, fused, per target ---
fig1, axs1 = plt.subplots(len(TARGETS), 2, figsize=(13, 4.5 * len(TARGETS)), sharex=True, squeeze=False)
for row, target in enumerate(TARGETS):
    fr = fused_results[target]
    for col, comp_name in enumerate(['East', 'North']):
        ax = axs1[row, col]
        ax.plot(fr['t'], fr['x1'][:, col], '--', color='C0', alpha=0.6, label=VIEW_REF)
        ax.plot(fr['t'], fr['x2'][:, col], '--', color='C1', alpha=0.6, label=VIEW_2)
        ax.plot(fr['t'], fr['x_fused'][:, col], '-', color='black', lw=2, label='Fused (BLUE)')
        ax.axhline(0, color='gray', lw=0.5)
        ax.set_title(f'Target {target} -- {comp_name}')
        ax.grid(alpha=0.3)
        ax.set_ylim(*pos_lims[col])
        if row == 0:
            ax.legend(fontsize=8)
axs1[-1, 0].set_xlabel('t (s)'); axs1[-1, 1].set_xlabel('t (s)')
plt.tight_layout()
save_fig_fusion(fig1, 'fig09a_position_fusion',
                f'East/North components: {VIEW_REF}, {VIEW_2} and fused (BLUE), per target.')


In [ ]:
# --- Per-frame sigma: View_1, View_2, fused ---
fig2, axs2 = plt.subplots(len(TARGETS), 2, figsize=(13, 4.5 * len(TARGETS)), sharex=True, squeeze=False)
for row, target in enumerate(TARGETS):
    fr = fused_results[target]
    for col, comp_name in enumerate(['sigma East', 'sigma North']):
        ax = axs2[row, col]
        ax.plot(fr['t'], np.sqrt(fr['C1'][:, col, col]) * 100, '--', color='C0', label=VIEW_REF)
        ax.plot(fr['t'], np.sqrt(fr['C2'][:, col, col]) * 100, '--', color='C1', label=VIEW_2)
        ax.plot(fr['t'], np.sqrt(fr['C_fused'][:, col, col]) * 100, '-', color='black', lw=2, label='Fused')
        ax.set_title(f'Target {target} -- {comp_name}')
        ax.set_ylabel('sigma (cm)')
        ax.grid(alpha=0.3)
        ax.set_ylim(*sigma_lims[col])
        if row == 0:
            ax.legend(fontsize=8)
axs2[-1, 0].set_xlabel('t (s)'); axs2[-1, 1].set_xlabel('t (s)')
plt.tight_layout()
save_fig_fusion(fig2, 'fig09_sigma_fusion',
                f'Per-frame standard deviation (East/North): {VIEW_REF}, {VIEW_2} and fused, per target.')


In [ ]:
# --- Plan-view trace: View_1, View_2, fused, per target ---
fig3, axs3 = plt.subplots(1, len(TARGETS), figsize=(7 * len(TARGETS), 7), squeeze=False)
axs3 = axs3[0]
for ax, target in zip(axs3, TARGETS):
    fr = fused_results[target]
    ax.plot(fr['x1'][:, 0], fr['x1'][:, 1], '--', color='C0', lw=1.2, alpha=0.6, label=VIEW_REF)
    ax.plot(fr['x2'][:, 0], fr['x2'][:, 1], '--', color='C1', lw=1.2, alpha=0.6, label=VIEW_2)
    ax.plot(fr['x_fused'][:, 0], fr['x_fused'][:, 1], '-', color='black', lw=2, label='Fused (BLUE)')
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_title(f'Plan-view trace -- Target {target}')
    ax.set_xlabel('East (m)'); ax.set_ylabel('North (m)')
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(*plan_xlim); ax.set_ylim(*plan_ylim)
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout()
save_fig_fusion(fig3, 'fig11_plan_fusion',
                'Fused plan-view trajectory of each target, compared with the single-view traces.')


In [ ]:
# --- Fused velocity (physical padding + marked windows) ---
fig4, axs4 = plt.subplots(1, len(TARGETS), figsize=(6.5 * len(TARGETS), 5.5), squeeze=False)
axs4 = axs4[0]
for ax, target in zip(axs4, TARGETS):
    fr = fused_results[target]
    x_fused_pad, t_fused_pad, pad_start_f, pad_end_f, n_pad_f = pad_only(fr['x_fused'], fr['t'], PAD_SEC)
    vel_fused_full = np.gradient(x_fused_pad, np.median(np.diff(t_fused_pad)), axis=0)

    ax.axvspan(*pad_start_f, color='gray', alpha=0.12, label='padding')
    ax.axvspan(*pad_end_f, color='gray', alpha=0.12)
    ax.plot(t_fused_pad, vel_fused_full[:, 0], '-', color='C0', label='vE fused')
    ax.plot(t_fused_pad, vel_fused_full[:, 1], '-', color='C1', label='vN fused')
    ax.axhline(0, color='k', lw=0.6)
    ax.set_title(f'Target {target} -- fused velocity')
    ax.set_xlabel('t (s)'); ax.set_ylabel('v (m/s)')
    ax.set_ylim(*vel_ylim)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    fused_results[target]['vel_full'] = vel_fused_full
    fused_results[target]['t_full'] = t_fused_pad
    v_max = np.linalg.norm(vel_fused_full, axis=1).max()
    print(f'Target {target} (fused): v_max = {v_max:.3f} m/s')
plt.tight_layout()
save_fig_fusion(fig4, 'fig12_velocity_fusion',
                'Fused (BLUE) velocity of each target, with padding windows marked.')


In [ ]:
# --- Uncertainty ellipses (2 sigma): View_1, View_2, fused, at 3 instants per target ---
N_SNAP = 3

fig5, axs5 = plt.subplots(1, len(TARGETS), figsize=(6 * len(TARGETS), 6), squeeze=False)
axs5 = axs5[0]
for ax, target in zip(axs5, TARGETS):
    fr = fused_results[target]
    idxs = np.linspace(0, len(fr['t']) - 1, N_SNAP).astype(int)
    for j, i in enumerate(idxs):
        add_cov_ellipse(ax, (0, 0), fr['C1'][i], facecolor='C0', alpha=0.12, edgecolor='C0',
                        lw=1.2, label=VIEW_REF if j == 0 else None)
        add_cov_ellipse(ax, (0, 0), fr['C2'][i], facecolor='C1', alpha=0.12, edgecolor='C1',
                        lw=1.2, label=VIEW_2 if j == 0 else None)
        add_cov_ellipse(ax, (0, 0), fr['C_fused'][i], facecolor='black', alpha=0.18, edgecolor='black',
                        lw=1.5, label='Fused' if j == 0 else None)
    ax.set_title(f'Target {target} -- uncertainty ellipses (2sigma, m)\n'
                f'{N_SNAP} instants: t={fr["t"][idxs[0]]:.1f}, {fr["t"][idxs[1]]:.1f}, {fr["t"][idxs[-1]]:.1f} s')
    ax.set_xlabel('E (m)'); ax.set_ylabel('N (m)')
    ax.axis('equal'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
save_fig_fusion(fig5, 'fig10_ellipses_fusion',
                f'Uncertainty ellipses (2 sigma) of {VIEW_REF}, {VIEW_2} and fused, at 3 instants per target.')


## 8. Save fused results

In [ ]:
for target in TARGETS:
    fr = fused_results[target]
    df_out = pd.DataFrame({
        't_s': fr['t'],
        'E_fused_m': fr['x_fused'][:, 0],
        'N_fused_m': fr['x_fused'][:, 1],
        'sigma_E_cm': np.sqrt(fr['C_fused'][:, 0, 0]) * 100,
        'sigma_N_cm': np.sqrt(fr['C_fused'][:, 1, 1]) * 100,
    })
    out_path = os.path.join(FUSION_DIR, f'fusion_BLUE_target_{target}.txt')
    df_out.to_csv(out_path, sep='\t', index=False)
    print(f'Saved: {out_path}')

print(f'\nLag applied ({VIEW_2} relative to {VIEW_REF}): {LAG_CONSENSUS:+.4f} s')
print(f'Figures and manifest in: {FUSION_FIGURES_DIR}')
